# 02 - Baseline Experiment
Compare the temperature-only Baseline model vs the Multi-Factor Risk Engine.
This notebook is the academic evidence of improvement.

In [ ]:
import sys
sys.path.insert(0, '..')

from src.data.ingestion import DataIngestor
from src.data.preprocessing import DataPreprocessor
from src.features.engineering import FeatureEngineer
from src.risk.baseline import BaselineRiskModel
from src.risk.risk_engine import MultiFactorRiskModel
from src.risk.confidence import ConfidenceEvaluator
from src.evaluation.metrics import compute_model_comparison
from src.evaluation.experiment import ExperimentRunner
import matplotlib.pyplot as plt
import pandas as pd

# 1. Build full pipeline
valid_df, _ = DataIngestor.load_synthetic_data(num_records=300, missing_rate=0.1)
raw_df = valid_df.copy()
preprocessor = DataPreprocessor()
preprocessor.fit(valid_df)
processed_df = preprocessor.transform(valid_df)
featured_df = FeatureEngineer().generate_features(processed_df)

# 2. Apply both models
df_with_baseline = BaselineRiskModel().calculate_risk(featured_df)
df_full = MultiFactorRiskModel().calculate_risk(df_with_baseline)
df_final = ConfidenceEvaluator().evaluate(raw_df, df_full)

print('Pipeline complete. Shape:', df_final.shape)

## Compare Risk Score Distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
df_final['baseline_risk_score'].hist(bins=25, ax=axes[0], color='#636EFA', edgecolor='black')
axes[0].set_title('Baseline Risk Score Distribution (Temperature Only)')
axes[0].set_xlabel('Score')
df_final['multi_factor_risk_score'].hist(bins=25, ax=axes[1], color='#EF553B', edgecolor='black')
axes[1].set_title('Multi-Factor Risk Score Distribution')
axes[1].set_xlabel('Score')
plt.suptitle('Baseline vs Multi-Factor: Score Distribution Comparison', fontsize=14)
plt.tight_layout()
plt.savefig('../data/processed/score_distributions.png', dpi=150)
plt.show()

## Category Comparison

In [ ]:
import numpy as np
baseline_cats = df_final['baseline_risk_category'].value_counts()
multifactor_cats = df_final['multi_factor_risk_category'].value_counts()

cats = ['LOW', 'MODERATE', 'HIGH', 'EXTREME']
x = np.arange(len(cats))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
bars1 = ax.bar(x - width/2, [baseline_cats.get(c, 0) for c in cats], width, label='Baseline', color='#636EFA')
bars2 = ax.bar(x + width/2, [multifactor_cats.get(c, 0) for c in cats], width, label='Multi-Factor', color='#EF553B')
ax.set_xticks(x)
ax.set_xticklabels(cats)
ax.set_ylabel('Number of Neighbourhoods')
ax.set_title('Risk Category Assignment: Baseline vs Multi-Factor Model')
ax.legend()
plt.tight_layout()
plt.savefig('../data/processed/category_comparison.png', dpi=150)
plt.show()

## Formal Metrics Comparison (Precision, Recall, F1)

In [ ]:
runner = ExperimentRunner()
result = runner.run_experiment(df_final, experiment_name='baseline_vs_multifactor')
runner.save_results(result)

metrics_df = pd.DataFrame(result['metrics']).T
print('\n=== Model Comparison Metrics ===')
print(metrics_df[['precision','recall','f1_score','accuracy']].to_string())
metrics_df[['precision','recall','f1_score','accuracy']].plot(kind='bar', figsize=(10, 6), colormap='Set2')
plt.title('Model Performance Metrics')
plt.xticks(rotation=0)
plt.ylim(0, 1.1)
plt.tight_layout()
plt.savefig('../data/processed/metrics_comparison.png', dpi=150)
plt.show()

## Key Findings

The Multi-Factor model is expected to produce higher **Recall** than the Baseline (fewer dangerous miss-classifications), while maintaining comparable Precision. The additional vulnerability and built-environment features help the model correctly identify areas that are hot **and** difficult to escape from.

In [ ]:
print('Baseline experiment complete. Results saved to data/processed/experiment_results.json')